In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RAW  = Path("../data/raw")
PROC = Path("../data/processed")

panel = pd.read_parquet(PROC / "balanced_panel.parquet")
panel["sensing_date"] = pd.to_datetime(panel.sensing_date)
panel["dow"] = panel.sensing_date.dt.dayofweek

print(f"{len(panel):,} rows, {panel.location_id.nunique()} sensors")

1,441,236 rows, 87 sensors


Step 0

Its predicted that removing piblic holidays will not affect the friday morning defecit, as the proportion of public holiday fridays to working fridays would be relatively low. Although foot traffic is exacerbated on some of the holidays particularly when there is events on such as footy matches, these matches also regularly take place in the afternoons and night time regardless of holidays.

It is predicted that the defecit measured on friday will be higher - i estimate 21%-24% - as not only would removing public holidays shift the baseline higher by increasing the average amount of people in the cbd from mon-thu, it may also decrease the amount of people in the cbd on fridays by removing high traffic times such as friday nefore afl finals. 

'''dgov = pd.read_csv(r'https://data.gov.au/data/dataset/b1bc6077-dadd-4f61-9f8c-002ab2cdff10/resource/33673aca-0857-42e5-b8f0-9981b4755686/download/australian-public-holidays-combined-2021-2025.csv')'''

In [ ]:
dgov.dtypes

ID                    int64
Date                float64
Holiday Name            str
Information             str
More Information        str
Jurisdiction            str
dtype: object

displayed the data types and made sure the ting was working

dgov = dgov[dgov["Jurisdiction"].str.lower() == 'vic'].copy()

dgov["Date"] = pd.to_datetime(dgov["Date"], format= '%Y%m%d')

the .to_datetime converted the date column in the dgov table above it into a proper format instead of the 7 digit float it was before

dgov = dgov[['Date',"Holiday Name"]]
dgov = dgov.rename(columns = {'Holiday Name': "Holiday"}).copy()

just saved the date and holiday name columns to remove clutter, and renamed holiday name to holiday for convenience

dgov['Source'] = 'data.gov.au'

dgov = dgov[(dgov['Date'] >= '2024-08-03')]
print(dgov.to_string())

one of the grand final fridays dropped. It was the only date error in all of the database, good thing it was spotted. 

bgov = pd.DataFrame({

    'Date': pd.to_datetime(["2025-09-26",'2026-01-01','2026-01-26','2026-03-09','2026-04-03','2026-04-04','2026-04-05','2026-04-06','2026-04-25','2026-06-08']),
    'Holiday': ['Friday before the AFL Grand Final','New Year\'s Day','Australia Day','Labour Day','Good Friday','Saturday before Easter Sunday','Easter Sunday','Easter Monday','ANZAC Day','King\'s Birthday' ],
    'Source': 'business.vic.gov.au'
})

In [ ]:
bgov.head()

,Date,Holiday,Source
0,2025-09-26,Friday before the AFL Grand Final,business.vic.gov.au
1,2026-01-01,New Year's Day,business.vic.gov.au
2,2026-01-26,Australia Day,business.vic.gov.au
3,2026-03-09,Labour Day,business.vic.gov.au
4,2026-04-03,Good Friday,business.vic.gov.au


In [ ]:
holidays = pd.concat([dgov,bgov], ignore_index=True)

holidays = (holidays 
            .drop_duplicates(subset='Date')
            .sort_values("Date")
            .reset_index(drop=True))

holidays['Holiday'] = holidays['Holiday'].str.strip()

print(holidays.to_string())

         Date                            Holiday               Source
0  2024-09-27  Friday before the AFL Grand Final          data.gov.au
1  2024-11-05                      Melbourne Cup          data.gov.au
2  2024-12-25                      Christmas Day          data.gov.au
3  2024-12-26                         Boxing Day          data.gov.au
4  2025-01-01                     New Year's Day          data.gov.au
5  2025-01-27                      Australia Day          data.gov.au
6  2025-03-10                         Labour Day          data.gov.au
7  2025-04-18                        Good Friday          data.gov.au
8  2025-04-19      Saturday before Easter Sunday          data.gov.au
9  2025-04-20                      Easter Sunday          data.gov.au
10 2025-04-21                      Easter Monday          data.gov.au
11 2025-04-25                          ANZAC Day          data.gov.au
12 2025-06-09                    King's Birthday          data.gov.au
13 2025-09-26  Frida

In [6]:
holidays = pd.read_csv(RAW / "Holidays.csv", parse_dates=["Date"])

normalised the data

Step 2

In [ ]:
holidays['Day-Name'] = holidays['Date'].dt.day_name()
holidays.groupby(holidays['Date'].dt.year).size()
holidays.to_csv(RAW / 'Holidays.csv', index=False)
holidays['Day-Name'].value_counts()

Day-Name
Monday       8
Friday       6
Thursday     3
Saturday     3
Tuesday      2
Wednesday    2
Sunday       2
Name: count, dtype: int64

Observation: now that the public holiday amounts and proportions can be observed, its visible that fridays are affected more(in that there are more public holidays) than the baseline (tue-thurs). This may mean that the prediction made may be incorrect. Additionally, mondays were more affected than fridays, which was unexpected.

Updated prediction: reflecting the fact that monday and friday lose a similar amount of days, the mon-fri comparison should not be affected by holidays, but given the nature of the different holidays this could be wrong. Given that the baseline is tue-thurs, and the baseline moves due to holidays having to be removed, i predict that the baseline traffic will increase by a small amount (2-5%). and for the friday vs baseline defecit, i imagine it will be pretty low(15%), revealing that the difference between firdays and the baseline is overestimated.  

Step 3

NameError: name 'bgov' is not defined